# PETA Age Attribute Recognition: Original Paper Baseline

Replicates the methodology of the original PETA benchmark (Deng et al., 2014): region-based color + texture histograms as features, one independent histogram-intersection-kernel SVM per attribute (one-vs-rest), evaluated with per-attribute mean Accuracy (mA). No CNNs, no deep learning, no ensembling - classical computer vision only.

Note: the original paper does not publish every implementation detail needed for a byte-exact reproduction (precise bin counts, region boundaries, SVM hyperparameter grid). This notebook is a faithful reconstruction of the documented method - same feature families, same classifier family, same evaluation metric - with the specific choices called out where the paper leaves them unspecified.

## 1. Setup

In [1]:
!pip install scikit-image -q


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)


## 2. Mount Drive and extract the dataset (Colab)

Skip this cell and just set `PETA_ROOT` directly if you're running locally instead of on Colab.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile

zip_path = "/content/drive/MyDrive/PETA.zip"
extract_path = "/content/PETA"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print(os.listdir(extract_path))


Mounted at /content/drive
['PETA dataset', 'ReadMe.txt']


## 3. Configuration

In [4]:
PETA_ROOT = "/content/PETA/PETA dataset"
SUBSETS = ["3DPeS", "CAVIAR4REID", "CUHK", "GRID", "i-LID", "MIT", "PRID", "SARC3D", "TownCentre", "VIPeR"]

AGE_CLASSES = ["Age16-30", "Age31-45", "Age46-60", "AgeAbove61"]
AGE_TO_IDX = {c: i for i, c in enumerate(AGE_CLASSES)}
NUM_CLASSES = len(AGE_CLASSES)

N_REGIONS = 4
COLOR_BINS = 16
LBP_POINTS = 16
LBP_RADII = [1, 2, 3]

TRAIN_FRAC = 9500 / 19000
VAL_FRAC = 1900 / 19000
TEST_FRAC = 7600 / 19000

C_GRID = [0.01, 0.1, 1, 10, 100]
GAMMA_GRID = [0.001, 0.01, 0.1, 1]

## 4. Building the dataframe from PETA label files

The raw age tags in Label.txt are `personalLess30`, `personalLess45`, `personalLess60`, `personalLarger60` (plus `personalLess15`, dropped since it doesn't fit any of the four target buckets).

In [5]:
RAW_AGE_TAGS = ["personalLess15", "personalLess30", "personalLess45", "personalLess60", "personalLarger60"]
RAW_TO_BUCKET = {
    "personalLess15": None,
    "personalLess30": "Age16-30",
    "personalLess45": "Age31-45",
    "personalLess60": "Age46-60",
    "personalLarger60": "AgeAbove61",
}


def parse_subset_labels(subset_name):
    subset_dir = os.path.join(PETA_ROOT, subset_name, "archive")
    label_path = os.path.join(subset_dir, "Label.txt")
    if not os.path.exists(label_path):
        return []

    with open(label_path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    records = []
    for line in lines:
        parts = line.split()
        pid = parts[0]
        attrs = set(parts[1:])

        raw_age = next((t for t in RAW_AGE_TAGS if t in attrs), None)
        if raw_age is None:
            continue

        age_label = RAW_TO_BUCKET[raw_age]
        if age_label is None:
            continue

        candidates = glob.glob(os.path.join(subset_dir, f"{pid}_*"))
        if not candidates:
            candidates = glob.glob(os.path.join(subset_dir, f"{pid}.*"))
        for img_path in candidates:
            records.append({
                "image_path": img_path,
                "age_label": age_label,
                "age_idx": AGE_TO_IDX[age_label],
            })
    return records


all_records = []
for s in SUBSETS:
    all_records.extend(parse_subset_labels(s))

df = pd.DataFrame(all_records)
print(df.shape)
df["age_label"].value_counts()


(14241, 3)


,count
age_label,
Age31-45,5804
Age16-30,5448
Age46-60,1840
AgeAbove61,1149


## 5. Train / val / test split

Uses the same proportions as PETA's official benchmark partition (9,500 / 1,900 / 7,600 out of 19,000), applied as a plain stratified random split — matching the original protocol, which does not group by identity.

In [6]:
train_df, temp_df = train_test_split(
    df, test_size=(1 - TRAIN_FRAC), stratify=df["age_idx"], random_state=SEED
)
rel_val = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
val_df, test_df = train_test_split(
    temp_df, test_size=(1 - rel_val), stratify=temp_df["age_idx"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))


7120 1424 5697


## 6. Feature extraction

Each image is split into three horizontal body regions (roughly head / torso / legs). Per region: 16-bin RGB and HSV color histograms plus a uniform LBP texture histogram, all L1-normalized. Regions are concatenated into one feature vector per image.

In [7]:
def region_split(img, n_regions=N_REGIONS):
    w, h = img.size
    step = h // n_regions
    regions = []
    for i in range(n_regions):
        top = i * step
        bottom = (i + 1) * step if i < n_regions - 1 else h
        regions.append(img.crop((0, top, w, bottom)))
    return regions


def color_histogram(region, bins=COLOR_BINS):
    rgb = np.array(region.convert("RGB")).astype(np.float32)
    hsv = np.array(region.convert("HSV")).astype(np.float32)

    hist = []
    for c in range(3):
        h, _ = np.histogram(rgb[:, :, c], bins=bins, range=(0, 255))
        hist.append(h / (h.sum() + 1e-8))
    for c in range(2):
        h, _ = np.histogram(hsv[:, :, c], bins=bins, range=(0, 255))
        hist.append(h / (h.sum() + 1e-8))
    return np.concatenate(hist)


def texture_histogram(region, points=LBP_POINTS, radii=None):
    if radii is None:
        radii = LBP_RADII
    gray = np.array(region.convert("L"))
    hists = []
    for r in radii:
        lbp = local_binary_pattern(gray, points, r, method="uniform")
        n_bins = points + 2
        h, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins))
        hists.append(h / (h.sum() + 1e-8))
    return np.concatenate(hists)


def extract_features(image_path):
    img = Image.open(image_path).convert("RGB")
    regions = region_split(img)
    feats = []
    for r in regions:
        feats.append(color_histogram(r))
        feats.append(texture_histogram(r))
    return np.concatenate(feats)

In [8]:
def build_feature_matrix(dataframe):
    return np.vstack([extract_features(p) for p in dataframe["image_path"]])


X_train = build_feature_matrix(train_df)
X_val = build_feature_matrix(val_df)
X_test = build_feature_matrix(test_df)

y_train = train_df["age_idx"].values
y_val = val_df["age_idx"].values
y_test = test_df["age_idx"].values

print(X_train.shape, X_val.shape, X_test.shape)


(7120, 550) (1424, 550) (5697, 550)


## 7. Histogram intersection kernel

The original benchmark uses an intersection-kernel SVM (IKSVM). Using the algebraic identity `min(a, b) = 0.5 * (a + b - |a - b|)`, the kernel can be computed in vectorized form via pairwise L1 distances instead of a slow nested loop.

In [9]:
def intersection_kernel(X, Y):
    sums_x = X.sum(axis=1)
    sums_y = Y.sum(axis=1)
    l1 = cdist(X, Y, metric="cityblock")
    return 0.5 * (sums_x[:, None] + sums_y[None, :]) - 0.5 * l1


K_train_train = intersection_kernel(X_train, X_train)
K_val_train = intersection_kernel(X_val, X_train)
K_test_train = intersection_kernel(X_test, X_train)


## 8. One binary IKSVM per age attribute

Each age bucket is trained as its own independent one-vs-rest binary classifier, matching the original benchmark's per-attribute framing rather than a single 4-way decision. `C` is selected per attribute from a small grid using validation mA.

In [10]:
def per_class_mA_single(y_true_bin, y_pred_bin):
    pos_mask = y_true_bin == 1
    neg_mask = y_true_bin == 0
    pos_acc = y_pred_bin[pos_mask].mean() if pos_mask.sum() > 0 else 0.0
    neg_acc = (y_pred_bin[neg_mask] == 0).mean() if neg_mask.sum() > 0 else 0.0
    return (pos_acc + neg_acc) / 2


def train_attribute_svm(bucket_idx):
    y_train_bin = (y_train == bucket_idx).astype(int)
    y_val_bin = (y_val == bucket_idx).astype(int)

    best_score, best_clf, best_config = -1, None, None

    for C in C_GRID:
        clf = SVC(kernel="precomputed", C=C, class_weight="balanced")
        clf.fit(K_train_train, y_train_bin)
        val_pred = clf.predict(K_val_train)
        mA = per_class_mA_single(y_val_bin, val_pred)
        if mA > best_score:
            best_score, best_clf, best_config = mA, clf, ("intersection", C, None)

    for C in C_GRID:
        for gamma in GAMMA_GRID:
            clf = SVC(kernel="rbf", C=C, gamma=gamma, class_weight="balanced")
            clf.fit(X_train, y_train_bin)
            val_pred = clf.predict(X_val)
            mA = per_class_mA_single(y_val_bin, val_pred)
            if mA > best_score:
                best_score, best_clf, best_config = mA, clf, ("rbf", C, gamma)

    return best_clf, best_config, best_score


classifiers = {}
configs = {}
for idx, cls in enumerate(AGE_CLASSES):
    clf, config, val_mA = train_attribute_svm(idx)
    classifiers[cls] = clf
    configs[cls] = config
    print(f"{cls}: config={config}, val_mA={val_mA:.4f}")

Age16-30: config=('rbf', 1, 1), val_mA=0.7996
Age31-45: config=('rbf', 10, 1), val_mA=0.7823
Age46-60: config=('rbf', 10, 0.1), val_mA=0.7897
AgeAbove61: config=('rbf', 10, 0.1), val_mA=0.9108


## 9. Test evaluation — per-attribute mean Accuracy (mA)

Same metric as the original benchmark table: each age bucket scored independently as a positive/negative decision, then averaged across positive-sample and negative-sample accuracy.

In [11]:
test_mA_scores = {}
for idx, cls in enumerate(AGE_CLASSES):
    y_test_bin = (y_test == idx).astype(int)
    kernel_type = configs[cls][0]
    if kernel_type == "intersection":
        test_pred = classifiers[cls].predict(K_test_train)
    else:
        test_pred = classifiers[cls].predict(X_test)
    mA = per_class_mA_single(y_test_bin, test_pred)
    test_mA_scores[cls] = mA
    print(f"{cls}: test_mA={mA:.4f}")

print("\naverage mA across age attributes:", np.mean(list(test_mA_scores.values())))

Age16-30: test_mA=0.8107
Age31-45: test_mA=0.7888
Age46-60: test_mA=0.8312
AgeAbove61: test_mA=0.9102

average mA across age attributes: 0.8352251330717204


In [12]:
from google.colab import files

print("Choose an .mp4 video from your laptop to upload...")
uploaded = files.upload()

INPUT_VIDEO_PATH = "/content/" + list(uploaded.keys())[0]
print("Uploaded file saved to:", INPUT_VIDEO_PATH)

Choose an .mp4 video from your laptop to upload...


Saving crowd_video2 (1).mp4 to crowd_video2 (1).mp4
Uploaded file saved to: /content/crowd_video2 (1).mp4


In [20]:
print("detect_people" in dir())
print("predict_age_for_crop" in dir())
print("region_split" in dir())
print("intersection_kernel" in dir())

False
False
True
True


In [21]:
@torch.no_grad()
def detect_people(frame_rgb):
    img_tensor = detector_transform(torch.from_numpy(frame_rgb).permute(2, 0, 1)).to(DEVICE)
    outputs = detector([img_tensor])[0]

    boxes = []
    for box, label, score in zip(outputs["boxes"], outputs["labels"], outputs["scores"]):
        if label.item() == COCO_PERSON_CLASS_ID and score.item() >= DETECTION_SCORE_THRESH:
            x1, y1, x2, y2 = box.cpu().numpy().astype(int)
            boxes.append((x1, y1, x2, y2))
    return boxes


def predict_age_for_crop(crop_img):
    feats = extract_features_from_image(crop_img)

    best_score, best_label = -1e9, None
    for cls in AGE_CLASSES:
        kernel_type = configs[cls][0]
        if kernel_type == "intersection":
            k = intersection_kernel(feats.reshape(1, -1), X_train)
            score = classifiers[cls].decision_function(k)[0]
        else:
            score = classifiers[cls].decision_function(feats.reshape(1, -1))[0]
        if score > best_score:
            best_score, best_label = score, cls
    return best_label


def extract_features_from_image(pil_img):
    regions = region_split(pil_img)
    feats = []
    for r in regions:
        feats.append(color_histogram(r))
        feats.append(texture_histogram(r))
    return np.concatenate(feats)

In [16]:
import cv2
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

detector_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=detector_weights)
detector.to(DEVICE).eval()

detector_transform = detector_weights.transforms()
COCO_PERSON_CLASS_ID = 1
DETECTION_SCORE_THRESH = 0.7

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 156MB/s]


In [22]:
from PIL import Image

OUTPUT_VIDEO_PATH = "/content/output_video.mp4"

cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

frame_idx = 0
while cap.isOpened():
    ret, frame_bgr = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    boxes = detect_people(frame_rgb)

    for (x1, y1, x2, y2) in boxes:
        crop = Image.fromarray(frame_rgb[y1:y2, x1:x2])
        if crop.width < 5 or crop.height < 5:
            continue
        age_label = predict_age_for_crop(crop)

        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame_bgr, age_label, (x1, max(y1 - 10, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    writer.write(frame_bgr)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"processed {frame_idx} frames")

cap.release()
writer.release()
print("done, saved to", OUTPUT_VIDEO_PATH)

processed 50 frames
processed 100 frames
processed 150 frames
processed 200 frames
processed 250 frames
processed 300 frames
processed 350 frames
processed 400 frames
processed 450 frames
processed 500 frames
processed 550 frames
processed 600 frames
processed 650 frames
processed 700 frames
processed 750 frames
processed 800 frames
processed 850 frames
processed 900 frames
processed 950 frames
processed 1000 frames
processed 1050 frames
processed 1100 frames
processed 1150 frames
processed 1200 frames
processed 1250 frames
processed 1300 frames
processed 1350 frames
processed 1400 frames
processed 1450 frames
processed 1500 frames
processed 1550 frames
processed 1600 frames
processed 1650 frames
processed 1700 frames
processed 1750 frames
processed 1800 frames
processed 1850 frames
processed 1900 frames
processed 1950 frames
processed 2000 frames
processed 2050 frames
processed 2100 frames
processed 2150 frames
processed 2200 frames
processed 2250 frames
processed 2300 frames
processe

In [23]:
print(classifiers)

{'Age16-30': SVC(C=1, class_weight='balanced', gamma=1), 'Age31-45': SVC(C=10, class_weight='balanced', gamma=1), 'Age46-60': SVC(C=10, class_weight='balanced', gamma=0.1), 'AgeAbove61': SVC(C=10, class_weight='balanced', gamma=0.1)}


In [24]:
from google.colab import files

print("Preparing download of the processed video...")
files.download(OUTPUT_VIDEO_PATH)

Preparing download of the processed video...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def predict_multiclass():
    n = len(y_test)
    scores = np.zeros((n, NUM_CLASSES))
    for idx, cls in enumerate(AGE_CLASSES):
        kernel_type = configs[cls][0]
        if kernel_type == "intersection":
            scores[:, idx] = classifiers[cls].decision_function(K_test_train)
        else:
            scores[:, idx] = classifiers[cls].decision_function(X_test)
    return scores.argmax(axis=1)


y_pred_multiclass = predict_multiclass()

overall_acc = accuracy_score(y_test, y_pred_multiclass)
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred_multiclass, average=None, labels=range(NUM_CLASSES)
)

print("Overall 4-way accuracy:", overall_acc)
print()

for idx, cls in enumerate(AGE_CLASSES):
    print(f"{cls}: precision={precision[idx]:.4f}  recall={recall[idx]:.4f}  f1={f1[idx]:.4f}  support={support[idx]}")

print()
print("Full classification report:")
print(classification_report(y_test, y_pred_multiclass, target_names=AGE_CLASSES))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_multiclass))

print()
print("Per-bucket mA (independent binary framing):")
for cls, score in test_mA_scores.items():
    print(f"{cls}: mA={score:.4f}")
print("Average mA:", np.mean(list(test_mA_scores.values())))

Overall 4-way accuracy: 0.766543794979814

Age16-30: precision=0.7302  recall=0.8123  f1=0.7691  support=2179
Age31-45: precision=0.7942  recall=0.7196  f1=0.7551  support=2322
Age46-60: precision=0.7302  recall=0.7283  f1=0.7293  support=736
AgeAbove61: precision=0.8966  recall=0.8478  f1=0.8715  support=460

Full classification report:
              precision    recall  f1-score   support

    Age16-30       0.73      0.81      0.77      2179
    Age31-45       0.79      0.72      0.76      2322
    Age46-60       0.73      0.73      0.73       736
  AgeAbove61       0.90      0.85      0.87       460

    accuracy                           0.77      5697
   macro avg       0.79      0.78      0.78      5697
weighted avg       0.77      0.77      0.77      5697

Confusion matrix:
[[1770  309   87   13]
 [ 535 1671   96   20]
 [  98   90  536   12]
 [  21   34   15  390]]

Per-bucket mA (independent binary framing):
Age16-30: mA=0.8107
Age31-45: mA=0.7888
Age46-60: mA=0.8312
AgeAbove6

In [26]:
import pickle
from google.colab import files

with open("/content/classifiers.pkl", "wb") as f:
    pickle.dump({"classifiers": classifiers, "configs": configs}, f)

files.download("/content/classifiers.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
import pickle

with open("/content/classifiers.pkl", "rb") as f:
    saved = pickle.load(f)

print(saved.keys())
print(saved["configs"])
print(saved["classifiers"])

dict_keys(['classifiers', 'configs'])
{'Age16-30': ('rbf', 1, 1), 'Age31-45': ('rbf', 10, 1), 'Age46-60': ('rbf', 10, 0.1), 'AgeAbove61': ('rbf', 10, 0.1)}
{'Age16-30': SVC(C=1, class_weight='balanced', gamma=1), 'Age31-45': SVC(C=10, class_weight='balanced', gamma=1), 'Age46-60': SVC(C=10, class_weight='balanced', gamma=0.1), 'AgeAbove61': SVC(C=10, class_weight='balanced', gamma=0.1)}


In [28]:
import os
import pickle
import shutil
from google.colab import files

export_dir = "/content/classifiers_export"
os.makedirs(export_dir, exist_ok=True)

with open(os.path.join(export_dir, "classifiers.pkl"), "wb") as f:
    pickle.dump(classifiers, f)

with open(os.path.join(export_dir, "configs.pkl"), "wb") as f:
    pickle.dump(configs, f)

shutil.make_archive("/content/classifiers_export", "zip", export_dir)

print("Zipped. Size:")
!ls -lh /content/classifiers_export.zip

files.download("/content/classifiers_export.zip")

Zipped. Size:
-rw-r--r-- 1 root root 34M Jul  8 14:01 /content/classifiers_export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>